# 03 - Evaluate Retrieval Separately

This notebook evaluates Chroma retrieval independently of answer generation.

Evaluation scope:
- Load both the base and challenging router datasets
- Evaluate retrieval on the correct expected local source where possible
- Compute Precision@k, Recall@k, Hit@k, and MRR when gold document ids exist
- Support challenging examples even when no gold document ids exist by exposing top-k retrieved documents for qualitative inspection
- Compare retrieval behavior across `top_k` values


## Learning Goal

Evaluate retrieval as its own component before judging answers. This lab teaches students to use document-level gold labels, top-k sweeps, and retrieval-specific metrics so they can distinguish retrieval failures from generation failures.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This lab assumes the expected source is known. That keeps the question focused: when we search the right collection, do we find the right evidence?

## Related AI Evals Concepts

- Don't Use Generic Eval Metrics: use Hit@k, Recall@k, Precision@k, and MRR instead of generic answer quality.
- Types Of Automated Evals: these are deterministic code-based checks against gold document ids.
- Don't Use Likert Scales: each document match is objective and verifiable.
- Synthetic Data: challenging examples can still be inspected qualitatively when gold ids are unavailable.


In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag')

In [13]:
import asyncio
import importlib
import os
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402
from agentic_rag.evaluation import parse_doc_ids, score_retrieval  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [14]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_FILE = TRACE_DIR / "03_retrieval_evaluation.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/otel_traces/03_retrieval_evaluation.jsonl')

## Load Retrieval Test Sets

The base dataset includes gold document ids for local retrieval examples. The challenging dataset is router-focused, so it does not include gold document ids; this notebook still retrieves for local-source examples and shows top-k evidence for inspection.


In [15]:
base_path = PROJECT_ROOT / "datasets/evaluation_dataset.csv"
challenge_path = PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv"

base_df = pd.read_csv(base_path)
challenge_df = pd.read_csv(challenge_path)

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"

base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

base_df.head()


,query,expected_source_type,expected_collection,expected_doc_ids,expected_answer,category,dataset
0,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA,medical_qna,0.0,The prevalence of X-linked chondrodysplasia punctata 1 is unknown. Several dozen affected males have been reported in the scientific literature.,frequency,base
1,What are the treatments for Kawasaki disease ?,Retrieve_QnA,medical_qna,1.0,These resources address the diagnosis or management of Kawasaki disease: - Cincinnati Children's Hospital Medical Center - Genetic Testing Registry: Acute febrile mucocutaneo...,treatment,base
2,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA,medical_qna,2.0,"Ellis-van Creveld syndrome can be caused by mutations in the EVC or EVC2 gene. Little is known about the function of these genes, although they appear to play important roles i...",genetic changes,base
3,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA,medical_qna,3.0,What are the signs and symptoms of Renal dysplasia-limb defects syndrome? The Human Phenotype Ontology provides the following list of signs and symptoms for Renal dysplasia-lim...,symptoms,base
4,What is (are) Fraser syndrome ?,Retrieve_QnA,medical_qna,4.0,Fraser syndrome is a rare disorder that affects development starting before birth. Characteristic features of this condition include eyes that are completely covered by skin an...,information,base


In [16]:
challenge_df.head()


,query,expected_source_type,category,rationale,dataset
0,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,device_plus_general_medical,"Mentions general risks, but the answer depends on a device manual contraindication.",challenging
1,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,explicit_source_meta_question,Asks about source selection and current outbreak updates rather than a stable disease answer.,challenging
2,What are the symptoms of Kawasaki disease and are there any new FDA-approved devices used to monitor it this year?,Web_Search,mixed_qna_recent_device,Contains stable symptoms plus recent device approval; current information should route to web.,challenging
3,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,mixed_device_qna,Primary entity is a specific device model and patient population/support question.,challenging
4,What changed recently in contraindications for dialysis machines from major manufacturers?,Web_Search,recent_device_external,"Device topic, but asks recent external/manufacturer changes.",challenging


In [17]:
dataset_summary = pd.concat(
    [
        base_df["expected_source_type"].value_counts().rename("base"),
        challenge_df["expected_source_type"].value_counts().rename("challenging"),
    ],
    axis=1,
).fillna(0).astype(int)

dataset_summary


,base,challenging
expected_source_type,,
Retrieve_QnA,12,1
Retrieve_Device,12,7
Web_Search,3,7


In [18]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


,collection,exists,count
0,medical_qna,True,16407
1,medical_device_manual,True,2694


## Define Retrieval-Specific Metrics


The CSV `expected_doc_ids` column stores local row numbers from the generated eval file, while Chroma returns collection document IDs such as `qna-7354` and `device-346`. Retrieval metrics compare IDs by exact string match, so this notebook resolves each gold row back to the Chroma document ID before scoring and keeps the original CSV value in `csv_expected_doc_ids` for auditability.


In [19]:
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}


def normalize_text(value: object) -> str:
    return " ".join(str(value or "").lower().split())


qna_id_by_question_answer = {
    (normalize_text(row["Question"]), normalize_text(row["Answer"])): f"qna-{row_index}"
    for row_index, row in qna_df.reset_index(drop=True).iterrows()
}
qna_ids_by_question = qna_df.reset_index(drop=True).groupby(
    qna_df["Question"].map(normalize_text)
).apply(lambda rows: [f"qna-{row_index}" for row_index in rows.index]).to_dict()

device_answer_columns = ["Indications_for_Use", "Contraindications", "Patient_Population"]
device_lookup_rows = []
for row_index, row in device_df.reset_index(drop=True).iterrows():
    for column in device_answer_columns:
        if column in row and not pd.isna(row[column]):
            device_lookup_rows.append(
                {
                    "doc_id": f"device-{row_index}",
                    "answer": normalize_text(row[column]),
                    "device_name": normalize_text(row["Device_Name"]),
                    "model_number": normalize_text(row["Model_Number"]),
                }
            )


def source_from_value(value: str) -> SourceType | None:
    try:
        source = SourceType(value)
    except ValueError:
        return None
    return source if source.value in LOCAL_SOURCES else None


def resolve_expected_doc_ids(row: pd.Series) -> list[str]:
    source = source_from_value(str(row["expected_source_type"]))
    query = normalize_text(row["query"])
    expected_answer = normalize_text(row.get("expected_answer", ""))

    if source == SourceType.RETRIEVE_QNA:
        exact_match = qna_id_by_question_answer.get((query, expected_answer))
        if exact_match:
            return [exact_match]
        return qna_ids_by_question.get(query, [])

    if source == SourceType.RETRIEVE_DEVICE:
        candidates = [item for item in device_lookup_rows if item["answer"] == expected_answer]
        model_matches = [item for item in candidates if item["model_number"] and item["model_number"] in query]
        if model_matches:
            return sorted({item["doc_id"] for item in model_matches})
        named_matches = [item for item in candidates if item["device_name"] and item["device_name"] in query]
        if named_matches:
            return sorted({item["doc_id"] for item in named_matches})
        return sorted({item["doc_id"] for item in candidates})

    return []


def expected_ids_from_row(row: pd.Series) -> list[str]:
    resolved_ids = resolve_expected_doc_ids(row)
    return resolved_ids or parse_doc_ids(row.get("expected_doc_ids", ""))


async def retrieve_one(row: pd.Series, top_k: int) -> dict[str, Any]:
    source = source_from_value(str(row["expected_source_type"]))
    csv_expected_doc_ids = parse_doc_ids(row.get("expected_doc_ids", ""))
    expected_doc_ids = expected_ids_from_row(row)

    if source is None:
        return {
            "dataset": row["dataset"],
            "query": row["query"],
            "expected_source_type": row["expected_source_type"],
            "retrieval_source": None,
            "top_k": top_k,
            "csv_expected_doc_ids": "|".join(csv_expected_doc_ids),
            "expected_doc_ids": "|".join(expected_doc_ids),
            "retrieved_doc_ids": "",
            "precision_at_k": None,
            "recall_at_k": None,
            "hit_at_k": None,
            "mrr": None,
            "top_doc_preview": None,
            "metric_status": "skipped_web_source",
        }

    retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
    docs = await retriever.retrieve(source, str(row["query"]))
    retrieved_doc_ids = [doc.doc_id for doc in docs]
    score = score_retrieval(retrieved_doc_ids, expected_doc_ids)
    has_gold = bool(expected_doc_ids)

    return {
        "dataset": row["dataset"],
        "query": row["query"],
        "expected_source_type": row["expected_source_type"],
        "retrieval_source": source.value,
        "top_k": top_k,
        "csv_expected_doc_ids": "|".join(csv_expected_doc_ids),
        "expected_doc_ids": "|".join(expected_doc_ids),
        "retrieved_doc_ids": "|".join(retrieved_doc_ids),
        "precision_at_k": score.precision_at_k if has_gold else None,
        "recall_at_k": score.recall_at_k if has_gold else None,
        "hit_at_k": score.hit_at_k if has_gold else None,
        "mrr": score.mrr if has_gold else None,
        "top_doc_preview": docs[0].text[:320] if docs else None,
        "metric_status": "scored" if has_gold else "qualitative_only_no_gold_doc_ids",
    }


async def evaluate_retrieval(df: pd.DataFrame, top_k_values: list[int]) -> pd.DataFrame:
    rows = []
    for top_k in top_k_values:
        rows.extend(await asyncio.gather(*(retrieve_one(row, top_k) for _, row in df.iterrows())))
    return pd.DataFrame(rows)


def summarize_retrieval(results: pd.DataFrame) -> pd.DataFrame:
    scored = results[results["metric_status"] == "scored"]
    if scored.empty:
        return pd.DataFrame()
    return (
        scored.groupby(["dataset", "expected_source_type", "top_k"], dropna=False)
        .agg(
            examples=("query", "count"),
            precision_at_k=("precision_at_k", "mean"),
            recall_at_k=("recall_at_k", "mean"),
            hit_at_k=("hit_at_k", "mean"),
            mrr=("mrr", "mean"),
        )
        .reset_index()
    )


## Score Retrieval Where Gold Documents Exist

This evaluates retrieval using the expected source collection, not the router-predicted source. That isolates Chroma retrieval quality from routing quality.


In [20]:
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
TOP_K_VALUES = [1, 3, 5, 10]

base_retrieval_results = await evaluate_retrieval(base_df, TOP_K_VALUES)
base_retrieval_results.head()


,dataset,query,expected_source_type,retrieval_source,top_k,csv_expected_doc_ids,expected_doc_ids,retrieved_doc_ids,precision_at_k,recall_at_k,hit_at_k,mrr,top_doc_preview,metric_status
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA,Retrieve_QnA,1,0,qna-7354,qna-7354,1.0,1.0,1.0,1.0,Question: How many people are affected by X-linked chondrodysplasia punctata 1 ?\nAnswer: The prevalence of X-linked chondrodysplasia punctata 1 is unknown. Several dozen affec...,scored
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA,Retrieve_QnA,1,1,qna-6837,qna-12125,0.0,0.0,0.0,0.0,Question: What are the treatments for Tubular aggregate myopathy ?\nAnswer: How might tubular aggregate myopathy be treated?,scored
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA,Retrieve_QnA,1,2,qna-8175,qna-8175,1.0,1.0,1.0,1.0,Question: What are the genetic changes related to Ellis-van Creveld syndrome ?\nAnswer: Ellis-van Creveld syndrome can be caused by mutations in the EVC or EVC2 gene. Little is...,scored
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA,Retrieve_QnA,1,3,qna-14533,qna-15015,0.0,0.0,0.0,0.0,Question: What are the symptoms of Bell's palsy ?\nAnswer: What are the symptoms of Bell's palsy?,scored
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA,Retrieve_QnA,1,4,qna-9958,qna-15453,0.0,0.0,0.0,0.0,"Question: What is (are) Cri du chat syndrome ?\nAnswer: Cri du chat syndrome, also known as 5p- (5p minus) syndrome or cat cry syndrome, is a genetic condition that is caused b...",scored


In [21]:
base_summary = summarize_retrieval(base_retrieval_results)
base_summary


,dataset,expected_source_type,top_k,examples,precision_at_k,recall_at_k,hit_at_k,mrr
0,base,Retrieve_Device,1,12,0.083333,0.083333,0.083333,0.083333
1,base,Retrieve_Device,3,12,0.055556,0.166667,0.166667,0.125000
2,base,Retrieve_Device,5,12,0.033333,0.166667,0.166667,0.125000
3,base,Retrieve_Device,10,12,0.025000,0.250000,0.250000,0.135417
4,base,Retrieve_QnA,1,12,0.250000,0.250000,0.250000,0.250000
5,base,Retrieve_QnA,3,12,0.138889,0.416667,0.416667,0.333333
6,base,Retrieve_QnA,5,12,0.100000,0.500000,0.500000,0.354167
7,base,Retrieve_QnA,10,12,0.058333,0.583333,0.583333,0.362500


In [22]:
base_scored = base_retrieval_results[base_retrieval_results["metric_status"] == "scored"]
base_failures = base_scored[base_scored["hit_at_k"] == 0].sort_values(["top_k", "expected_source_type"])
base_failures[["query", "expected_source_type", "top_k", "csv_expected_doc_ids", "expected_doc_ids", "retrieved_doc_ids", "top_doc_preview"]].head(20)


,query,expected_source_type,top_k,csv_expected_doc_ids,expected_doc_ids,retrieved_doc_ids,top_doc_preview
12,What are the contraindications for the Model 1606 Electrosurgical Unit?,Retrieve_Device,1,0,device-346,device-1291,Device_Name: Electrosurgical Unit\nModel_Number: EDW280\nManufacturer: Edwards Lifesciences\nPatient_Population: All\nIndications_for_Use: Intended for sterilization guidance d...
13,What is the intended use of the 3M Healthcare Dialysis Machine model X-6538?,Retrieve_Device,1,1,device-1282,device-620,Device_Name: Dialysis Machine\nModel_Number: Model 7546\nManufacturer: Johnson & Johnson\nPatient_Population: Pediatric (2-18)\nIndications_for_Use: Intended for infection prev...
14,Which patient population is the Ventilator model SON230 for?,Retrieve_Device,1,2,device-868,device-450,Device_Name: Pacemaker\nModel_Number: Pro245\nManufacturer: Abbott\nPatient_Population: Adult and Pediatric\nIndications_for_Use: Used for post-operative electrotherapy managem...
15,What are the contraindications for the Max787 Dialysis Machine?,Retrieve_Device,1,3,device-1850,device-179,Device_Name: Dialysis Machine\nModel_Number: Model 1703\nManufacturer: Edwards Lifesciences\nPatient_Population: Adult\nIndications_for_Use: Used for intraoperative measurement...
16,What is the intended use of the Abbott Electrosurgical Unit model Plus691?,Retrieve_Device,1,4,device-352,device-2225,Device_Name: Ultrasound Scanner\nModel_Number: Model 749\nManufacturer: Nipro Corporation\nPatient_Population: Neonatal\nIndications_for_Use: Intended for alarm management supp...
17,Which patient population is the Defibrillator model Z-2047 for?,Retrieve_Device,1,5,device-156,device-1757,Device_Name: Defibrillator\nModel_Number: Model 7768\nManufacturer: Fresenius Medical Care\nPatient_Population: Infant (0-2)\nIndications_for_Use: Intended for alarm management...
18,What are the contraindications for the Max432 X-Ray Machine?,Retrieve_Device,1,6,device-1457,device-1188,Device_Name: X-Ray Machine\nModel_Number: GET342\nManufacturer: Getinge\nPatient_Population: Adult (18-65)\nIndications_for_Use: Intended for quality control guidance during mi...
19,What is the intended use of the Beckman Coulter Nebulizer model BEC582?,Retrieve_Device,1,7,device-2236,device-2225,Device_Name: Ultrasound Scanner\nModel_Number: Model 749\nManufacturer: Nipro Corporation\nPatient_Population: Neonatal\nIndications_for_Use: Intended for alarm management supp...
21,What are the contraindications for the C-5476 Stent?,Retrieve_Device,1,9,9,device-1314,Device_Name: Surgical Drill\nModel_Number: EDW745\nManufacturer: Edwards Lifesciences\nPatient_Population: Pediatric\nIndications_for_Use: Indicated for intracranial pressure o...
22,What is the intended use of the Thermo Fisher Scientific Surgical Drill model Model 6091?,Retrieve_Device,1,10,device-1237,device-1919,Device_Name: Surgical Robot\nModel_Number: Model 5314\nManufacturer: Philips Healthcare\nPatient_Population: Pediatric (2-18)\nIndications_for_Use: Used for intraoperative cali...


## Inspect Hard Examples Without Gold Documents

The challenging dataset has route labels and rationales but no gold Chroma document ids. For local-source examples, this section retrieves top-k documents from the expected local source and marks the row as qualitative-only. `Web_Search` rows are skipped because they do not use Chroma.


In [23]:
challenge_retrieval_results = await evaluate_retrieval(challenge_df, [3])
challenge_retrieval_results.head()


,dataset,query,expected_source_type,retrieval_source,top_k,csv_expected_doc_ids,expected_doc_ids,retrieved_doc_ids,precision_at_k,recall_at_k,hit_at_k,mrr,top_doc_preview,metric_status
0,challenging,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Retrieve_Device,3,,,device-560|device-1628|device-786,None,None,None,None,Device_Name: Ultrasound Scanner\nModel_Number: Plus388\nManufacturer: Roche\nPatient_Population: Adult (>65)\nIndications_for_Use: Intended for surgical neurological procedures...,qualitative_only_no_gold_doc_ids
1,challenging,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,NaN,3,,,,None,None,None,None,NaN,skipped_web_source
2,challenging,What are the symptoms of Kawasaki disease and are there any new FDA-approved devices used to monitor it this year?,Web_Search,NaN,3,,,,None,None,None,None,NaN,skipped_web_source
3,challenging,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,Retrieve_Device,3,,,device-1194|device-1858|device-1096,None,None,None,None,Device_Name: Electrosurgical Unit\nModel_Number: BEC965\nManufacturer: Beckman Coulter\nPatient_Population: All\nIndications_for_Use: Used for emergency hemodialysis in acute s...,qualitative_only_no_gold_doc_ids
4,challenging,What changed recently in contraindications for dialysis machines from major manufacturers?,Web_Search,NaN,3,,,,None,None,None,None,NaN,skipped_web_source


In [24]:
challenge_retrieval_results["metric_status"].value_counts().rename_axis("status").reset_index(name="count")


,status,count
0,qualitative_only_no_gold_doc_ids,8
1,skipped_web_source,7


In [25]:
challenge_local_inspection = challenge_retrieval_results[
    challenge_retrieval_results["metric_status"] == "qualitative_only_no_gold_doc_ids"
][["query", "expected_source_type", "retrieved_doc_ids", "top_doc_preview"]]

challenge_local_inspection


,query,expected_source_type,retrieved_doc_ids,top_doc_preview
0,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,device-560|device-1628|device-786,Device_Name: Ultrasound Scanner\nModel_Number: Plus388\nManufacturer: Roche\nPatient_Population: Adult (>65)\nIndications_for_Use: Intended for surgical neurological procedures...
3,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,device-1194|device-1858|device-1096,Device_Name: Electrosurgical Unit\nModel_Number: BEC965\nManufacturer: Beckman Coulter\nPatient_Population: All\nIndications_for_Use: Used for emergency hemodialysis in acute s...
5,Compare the contraindications of the Max787 Dialysis Machine with standard contraindications for dialysis patients.,Retrieve_Device,device-1648|device-1565|device-2156,Device_Name: Surgical Drill\nModel_Number: Pro721\nManufacturer: Synthes\nPatient_Population: Pediatric (2-18)\nIndications_for_Use: Intended for continuous monitoring evaluati...
6,What is Fraser syndrome and can any surgical robot in our manuals be used for related procedures?,Retrieve_Device,device-324|device-792|device-228,Device_Name: Surgical Robot\nModel_Number: Max578\nManufacturer: Boston Scientific\nPatient_Population: Adult and Pediatric\nIndications_for_Use: Used for post-operative chemot...
7,Which device should be used for emergency pulmonary stabilization in pediatric patients?,Retrieve_Device,device-561|device-492|device-1959,Device_Name: Catheter\nModel_Number: Pro753\nManufacturer: Karl Storz\nPatient_Population: Adult and Pediatric\nIndications_for_Use: Used for long-term fluid resuscitation deli...
10,What is the sterilization method for devices indicated for oncology treatment centers?,Retrieve_Device,device-108|device-1360|device-1226,Device_Name: CPAP Machine\nModel_Number: Model 7163\nManufacturer: Align Technology\nPatient_Population: Geriatric\nIndications_for_Use: Used for phototherapy guidance in oncol...
12,What are common symptoms after using a nebulizer incorrectly?,Retrieve_QnA,qna-12392|qna-4048|qna-10842,Question: What are the symptoms of Tarsal tunnel syndrome ?\nAnswer: What symptoms are commonly seen in tarsal tunnel syndrome? The symptoms of tarsal tunnel syndrome can vary ...
13,Can a blood pressure monitor be used for autoimmune disorder patients according to the manual?,Retrieve_Device,device-347|device-701|device-1005,Device_Name: Blood Pressure Monitor\nModel_Number: Model 3388\nManufacturer: Cook Medical\nPatient_Population: Neonatal\nIndications_for_Use: Used for thermal therapy guidance ...


## Pay Attention To

- Retrieval evals should be separated from answer evals so students can locate the failure.
- Rows without gold document ids should not be averaged into retrieval metrics.
- Increasing `top_k` can improve recall while lowering precision; this tradeoff matters in RAG.
- Qualitative inspection is still valuable when an example lacks a clean gold label.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against both the base and challenging router datasets. It reads `OPENAI_API_KEY` from the active environment only; it does not load `.env`.


In [26]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_router_results = pd.DataFrame()
llm_router_summary = pd.DataFrame()


def load_router_eval_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Query" in df.columns:
        df = df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})
    df = df.copy()
    df["dataset"] = dataset_name
    df["expected_source_type"] = df["expected_source_type"].astype(str)
    return df


async def evaluate_llm_router_dataset(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(str(query)) for query in df["query"].tolist()))
    results = df.copy()
    results["router_mode"] = "llm"
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    keep_columns = [
        "dataset",
        "router_mode",
        "query",
        "expected_source_type",
        "predicted_source_type",
        "route_correct",
        "category",
        "rationale",
    ]
    return results[[column for column in keep_columns if column in results.columns]]


def summarize_llm_router_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    return results.groupby(["dataset", "router_mode"], dropna=False).agg(
        examples=("query", "count"),
        accuracy=("route_correct", "mean"),
        failures=("route_correct", lambda values: int((~values).sum())),
    ).reset_index()


if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    llm_router = QueryRouter(
        generator=OpenAITextGenerator(api_key=api_key, model=LLM_ROUTER_MODEL, timeout=LLM_ROUTER_TIMEOUT),
        mode="llm",
    )
    llm_router_base_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/evaluation_dataset.csv", "base")
    llm_router_challenge_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv", "challenging")
    llm_router_results = pd.concat(
        [
            await evaluate_llm_router_dataset(llm_router, llm_router_base_df),
            await evaluate_llm_router_dataset(llm_router, llm_router_challenge_df),
        ],
        ignore_index=True,
    )
    llm_router_summary = summarize_llm_router_results(llm_router_results)

llm_router_summary


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.0,0
1,challenging,llm,15,0.8,3


In [27]:
if not llm_router_results.empty:
    display(llm_router_summary)
    display(pd.crosstab(
        [llm_router_results["dataset"], llm_router_results["expected_source_type"]],
        llm_router_results["predicted_source_type"],
        dropna=False,
    ))
    display(llm_router_results.loc[~llm_router_results["route_correct"]])


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.0,0
1,challenging,llm,15,0.8,3


predicted_source_type             Retrieve_Device  Retrieve_QnA  Web_Search
dataset     expected_source_type                                           
base        Retrieve_Device                    12             0           0
            Retrieve_QnA                        0            12           0
            Web_Search                          0             0           3
challenging Retrieve_Device                     5             2           0
            Retrieve_QnA                        0             1           0
            Web_Search                          0             1           6

,dataset,router_mode,query,expected_source_type,predicted_source_type,route_correct,category,rationale
27,challenging,llm,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Retrieve_QnA,False,device_plus_general_medical,"Mentions general risks, but the answer depends on a device manual contraindication."
28,challenging,llm,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,Retrieve_QnA,False,explicit_source_meta_question,Asks about source selection and current outbreak updates rather than a stable disease answer.
33,challenging,llm,What is Fraser syndrome and can any surgical robot in our manuals be used for related procedures?,Retrieve_Device,Retrieve_QnA,False,mixed_qna_device,"Contains a disease definition, but asks whether manuals contain a suitable device."


## Export A Combined Retrieval Artifact

The combined output keeps scored base examples and qualitative challenging examples in the same schema, which makes it easier for later notebooks to compare retrieval behavior.


In [28]:
combined_retrieval_results = pd.concat(
    [base_retrieval_results, challenge_retrieval_results],
    ignore_index=True,
)

combined_retrieval_results["metric_status"].value_counts().rename_axis("status").reset_index(name="count")


,status,count
0,scored,96
1,skipped_web_source,19
2,qualitative_only_no_gold_doc_ids,8


In [29]:
output_path = PROJECT_ROOT / "output/retrieval_evaluation_results.csv"
summary_path = PROJECT_ROOT / "output/retrieval_evaluation_summary.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.parent.mkdir(parents=True, exist_ok=True)
combined_retrieval_results.to_csv(output_path, index=False)
base_summary.to_csv(summary_path, index=False)

output_path, summary_path


(PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/retrieval_evaluation_results.csv'),
 PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/retrieval_evaluation_summary.csv'))